# Retrieval and Ranking

**Stage 1** — Bi-encoder dense retrieval via FAISS
- Top-5 standalone result -> `retrieval_biencoder_top5.jsonl`
- Top-20 candidate pool -> `retrieval_biencoder_top20.jsonl` (input for Stage 2 only)

**Stage 2** — Cross-encoder reranking (reads top-20, outputs top-5)
- Reranked result -> `retrieval_reranked_top5.jsonl`

In [216]:
# !pip install sentence-transformers

In [98]:
import json
import os
import time
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from typing import List, Dict

In [114]:
# Config
BENCHMARK_PATH = "../data/benchmark_dataset.json"
FILE_NAME="chunks_structure_aware_fine"
EMBEDDED_PATH = f"../embedding/output/{FILE_NAME}/embedded_minilm.jsonl"
FAISS_PATH = f"../embedding/output/{FILE_NAME}/embedded_minilm.faiss"

MODEL_NAME = "all-MiniLM-L6-v2"

RESULTS_DIR = "results/"
TOP20_PATH = os.path.join(RESULTS_DIR, f"retrieval_biencoder_top20_{FILE_NAME}_benchmark_dataset.jsonl")
TOP5_PATH = os.path.join(RESULTS_DIR, f"retrieval_biencoder_top5_{FILE_NAME}_benchmark_dataset.jsonl")
RERANKED_PATH = os.path.join(RESULTS_DIR, f"retrieval_reranked_top5_{FILE_NAME}_benchmark_dataset.jsonl")
os.makedirs(RESULTS_DIR, exist_ok=True)

In [115]:
def load_benchmark(path: str) -> List[Dict]:
    """
    Load benchmark JSON (nested source-grouped format):
      {"sources": [{"source": "...", "questions": [{"question": ..., "answer": ...}]}]}
    Returns flat list with keys: query_id, query, answer, source.
    """
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    flat = []
    idx = 1
    for src_block in data["sources"]:
        source = src_block["source"]
        for q in src_block["questions"]:
            flat.append({
                "query_id": f"q_{idx:03d}",
                "query":    q["question"],
                "answer":   q["answer"],
                "source":   source,
            })
            idx += 1
    return flat


# Load benchmark
benchmark = load_benchmark(BENCHMARK_PATH)
print(f"Loaded {len(benchmark)} benchmark queries")

# Load FAISS index + corpus lines
index = faiss.read_index(FAISS_PATH)
with open(EMBEDDED_PATH, "r", encoding="utf-8") as f:
    corpus_lines = f.readlines()
assert len(corpus_lines) == index.ntotal
print(f"FAISS index: {index.ntotal} vectors, dim={index.d}")

# Load bi-encoder
import nest_asyncio
nest_asyncio.apply()

bi_encoder = SentenceTransformer(MODEL_NAME)
print(f"Bi-encoder: {MODEL_NAME}")


Loaded 72 benchmark queries
FAISS index: 2444 vectors, dim=384


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Bi-encoder: all-MiniLM-L6-v2


In [116]:
def retrieve_biencoder(
    queries: List[Dict],
    index: faiss.Index,
    corpus_lines: List[str],
    model: SentenceTransformer,
    top_n: int,
) -> List[Dict]:
    """Bi-encoder retrieval for all queries. Returns top-n candidates per query."""
    query_texts = [q["query"] for q in queries]
    query_embs = model.encode(
        query_texts, normalize_embeddings=True, show_progress_bar=True
    ).astype(np.float32)

    scores_all, indices_all = index.search(query_embs, top_n)

    results = []
    for q, scores, indices in zip(queries, scores_all, indices_all):
        retrieved = []
        for rank, (score, idx) in enumerate(zip(scores, indices), start=1):
            if idx == -1:
                continue
            chunk = json.loads(corpus_lines[idx])
            retrieved.append({
                "rank": rank,
                "chunk_id": chunk["chunk_id"],
                "score": round(float(score), 6),
                "text": chunk["text"],
            })
        results.append({
            "query_id": q["query_id"],
            "query": q["query"],
            "retrieved": retrieved,
        })
    return results


def save_results(results: List[Dict], path: str):
    with open(path, "w", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"Saved {len(results)} queries to {path}")


---

## Stage 1A: Top-5 (standalone)

Final retrieval result when no reranker is used. Ready for prompting and evaluation.

In [117]:
print(TOP5_PATH)

results/retrieval_biencoder_top5_chunks_structure_aware_fine_benchmark_dataset.jsonl


In [118]:
t0 = time.time()
results_top5 = retrieve_biencoder(benchmark, index, corpus_lines, bi_encoder, top_n=5)
print(f"Done: {len(results_top5)} queries, {time.time()-t0:.2f}s")

save_results(results_top5, TOP5_PATH)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Done: 72 queries, 0.45s
Saved 72 queries to results/retrieval_biencoder_top5_chunks_structure_aware_fine_benchmark_dataset.jsonl


In [119]:
# Inspect first query
sample = results_top5[0]
print(f"Query: {sample['query']}\n")
for r in sample["retrieved"]:
    print(f"  #{r['rank']}  score={r['score']:.4f}  {r['chunk_id']}")
    print(f"       {r['text'][:100]}...\n")

Query: What does the word 'bibimbap' literally mean in Korean?

  #1  score=0.6678  wiki_0006_chunk_0
       Bibimbap ( BEE-bim-bap; Korean: 비빔밥; lit. 'mixed rice'), sometimes romanised as bi bim bap or bi bim...

  #2  score=0.6175  wikibooks_0069_chunk_0
       Bibimbap (also called Bibimba or Bibimbop) is a typical Korean meal that literally means "mixed rice...

  #3  score=0.5669  wiki_0006_chunk_2
       The hangul transcription beubwieum (브뷔음) first appears in the 1810 encyclopedia Mongyupyeon by Jang ...

  #4  score=0.5565  wiki_0006_chunk_10
       Some people attach symbolism to the ingredients of bibimbap. Black or dark colours represent north a...

  #5  score=0.5440  wiki_0006_chunk_3
       The origin of bibimbap is unknown. Some scholars assert that bibimbap originates from the traditiona...



---

## Stage 1B: Top-20 (for Stage 2 reranker)


It is a wider candidate pool so the cross-encoder reranker can pick better top-5 from 20 candidates instead of just reordering 5.

In [120]:
t0 = time.time()
results_top20 = retrieve_biencoder(benchmark, index, corpus_lines, bi_encoder, top_n=20)
print(f"Done: {len(results_top20)} queries, {time.time()-t0:.2f}s")

save_results(results_top20, TOP20_PATH)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Done: 72 queries, 0.72s
Saved 72 queries to results/retrieval_biencoder_top20_chunks_structure_aware_fine_benchmark_dataset.jsonl


---
---

## Stage 2: Cross-Encoder Reranking

Read the top-20 candidates above, rerank with a cross-encoder, output top-5.


In [121]:
from sentence_transformers import CrossEncoder

In [122]:
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

TOP_K = 5

In [123]:
cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL)
print(f"Cross-encoder loaded: {CROSS_ENCODER_MODEL}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cross-encoder loaded: cross-encoder/ms-marco-MiniLM-L-6-v2


In [124]:
with open(TOP20_PATH, "r", encoding="utf-8") as f:
    results_top20 = [json.loads(line) for line in f]
print(f"Loaded {len(results_top20)} queries from {TOP20_PATH}")

Loaded 72 queries from results/retrieval_biencoder_top20_chunks_structure_aware_fine_benchmark_dataset.jsonl


In [125]:
def rerank(
    stage1_results: List[Dict],
    cross_encoder: CrossEncoder,
    top_k: int = TOP_K,
) -> List[Dict]:
    """
    Re-rank Stage 1 candidates with a Cross-Encoder.

    For each query:
      1. Build (query, chunk_text) pairs for all top-20 candidates.
      2. Score every pair in one batch through the Cross-Encoder.
      3. Sort by Cross-Encoder score (descending).
      4. Return the top-k results with updated ranks.

    The original bi-encoder score is preserved as `score_biencoder`
    and the Cross-Encoder logit is stored as `score`.
    """
    reranked = []
    for item in stage1_results:
        query = item["query"]
        candidates = item["retrieved"]

        if not candidates:
            reranked.append(item)
            continue

        pairs = [(query, c["text"]) for c in candidates]
        ce_scores = cross_encoder.predict(pairs)

        scored = sorted(
            zip(ce_scores, candidates),
            key=lambda x: x[0],
            reverse=True,
        )

        top_k_hits = []
        for new_rank, (ce_score, cand) in enumerate(scored[:top_k], start=1):
            top_k_hits.append({
                "rank":            new_rank,
                "chunk_id":        cand["chunk_id"],
                "score":           cand["score"],
                "rerank_score":    round(float(ce_score), 6),
                "text":            cand["text"],
            })

        reranked.append({
            "query_id":  item["query_id"],
            "query":     query,
            "retrieved": top_k_hits,
        })
    return reranked

In [126]:
t0 = time.time()
results_reranked = rerank(results_top20, cross_encoder)
print(f"Done: {len(results_reranked)} queries, {time.time()-t0:.2f}s")

save_results(results_reranked, RERANKED_PATH)

Done: 72 queries, 83.75s
Saved 72 queries to results/retrieval_reranked_top5_chunks_structure_aware_fine_benchmark_dataset.jsonl


In [127]:
sample = results_reranked[0]
print(f"Query: {sample['query']}\n")
for r in sample["retrieved"]:
    print(f"  #{r['rank']}  ce={r['score']:.4f}  bi={r['rerank_score']:.4f}  {r['chunk_id']}")
    print(f"       {r['text'][:100]}...\n")

Query: What does the word 'bibimbap' literally mean in Korean?

  #1  ce=0.6175  bi=7.6197  wikibooks_0069_chunk_0
       Bibimbap (also called Bibimba or Bibimbop) is a typical Korean meal that literally means "mixed rice...

  #2  ce=0.6678  bi=6.7688  wiki_0006_chunk_0
       Bibimbap ( BEE-bim-bap; Korean: 비빔밥; lit. 'mixed rice'), sometimes romanised as bi bim bap or bi bim...

  #3  ce=0.4945  bi=5.6782  blog_0022_chunk_0
       Bibimbap literally means “mixed rice” and describes a dish of cooked white rice topped with varietie...

  #4  ce=0.4676  bi=5.0531  wiki_0006_chunk_8
       Hoedeopbap (회덮밥) is a bibimbap with a variety of raw seafood, such as olive flounder, salmon, tuna o...

  #5  ce=0.4808  bi=4.4839  wiki_0006_chunk_1
       The earliest names for bibimbap appear in Korean hanja texts. Its first name was hondon-ban (混沌飯). T...



### Rank-change analysis

How much does the Cross-Encoder actually reorder the bi-encoder's top-5?

In [128]:
top1_changed = 0
rank_shifts   = []

for s1, s2 in zip(results_top5, results_reranked):
    s1_ids = [h["chunk_id"] for h in s1["retrieved"]]
    s2_ids = [h["chunk_id"] for h in s2["retrieved"]]

    if s1_ids and s2_ids and s1_ids[0] != s2_ids[0]:
        top1_changed += 1

    for new_rank, chunk_id in enumerate(s2_ids, start=1):
        if chunk_id in s1_ids:
            old_rank = s1_ids.index(chunk_id) + 1
            rank_shifts.append(abs(new_rank - old_rank))

n = len(results_top5)
avg_shift = sum(rank_shifts) / max(len(rank_shifts), 1)
print(f"Top-1 changed by reranking : {top1_changed}/{n} ({top1_changed/n*100:.0f}%)")
print(f"Avg absolute rank shift    : {avg_shift:.2f} positions")

Top-1 changed by reranking : 26/72 (36%)
Avg absolute rank shift    : 0.93 positions
